<a href="https://colab.research.google.com/github/cbikash/movie-recommendation-system/blob/main/movieReco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import json

movies = pd.read_csv('https://raw.githubusercontent.com/cbikash/movie-recommendation-system/refs/heads/main/tmdb_5000_movies.csv')
credits = pd.read_csv("https://raw.githubusercontent.com/cbikash/movie-recommendation-system/refs/heads/main/tmdb_5000_credits.csv")
movies.head(1)

In [ ]:
movies.info()

In [ ]:
movies.describe()

In [ ]:
credits.info()

In [ ]:
credits.head(1)

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
# useful table columns for making tags
#genres
#id
#keywords
#overview
#title
#cast
#crew

movies = movies[['genres', 'id', 'keywords', 'overview', 'title', 'cast', 'crew']]
movies.head(2)

In [ ]:
print(movies.isnull().sum())

In [ ]:
movies.dropna(inplace=True) #we will drop overview because there is only 3 data and overview is the important columns
print(movies.isnull().sum())

In [ ]:
movies.duplicated().sum()

In [ ]:
movies['genres'][0]

In [ ]:
def convert(values):
  l = []
  for o in json.loads(values):
    l.append(o['name'])
  return l

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['cast'][0]

In [ ]:
def convert2(values):
  l = []
  counter = 0
  for o in json.loads(values):
    if counter != 3:
      l.append(o['name'])
    else:
      break;
    counter = counter + 1;
  return l

In [ ]:
movies['cast'] = movies['cast'].apply(convert2)

In [ ]:

def covert_crew(objects):
  d = []
  for crew in json.loads(objects):
    if(crew['job'] == 'Director'):
      d.append(crew['name'])
      break;
  return d

In [ ]:
movies['crew'] = movies['crew'].apply(covert_crew)

In [ ]:
movies.head(5)

In [ ]:
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ", "") for i in x])


In [ ]:
movies.head(5)

In [ ]:
movies['tags'] = movies['genres'] + movies['overview'] + movies['cast'] + movies['crew'] + movies['keywords']

In [ ]:
movies['tags']

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x:" ".join(x))
new_df = movies[['id', 'title', 'tags']]

new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [ ]:
new_df
import nltk
from nltk.stem.porter import  PorterStemmer
stem = PorterStemmer()

In [ ]:
def stem_porter(text):
  y = []
  for i in text.split():
    y.append(stem.stem(i))
  return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem_porter)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vector = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vector

In [ ]:
cv.get_feature_names_out()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(vector)
similarities[1]

In [ ]:
def recommend(movie):
  movie_index = new_df[new_df['title'] == movie].index[0]
  distances = similarities[movie_index]
  movie_list = sorted(list(enumerate(distances)), reverse=True, key= lambda x:x[1])[1:6]
  print(distances)

  for i in movie_list:
    movie_name = new_df.iloc[i[0]].title
    print(movie_name)

recommend('Avatar')